In [ ]:
import pandas as pd

panel = pd.read_csv("D:\\ENSAE Paris\\3A\\S1\\ML_Climate_risk\\PROJET\\29f9885375e0bb8047745286a114a491\\catnat_drought\\catnat_drought\\panel_final.csv", sep=";")
panel.head()

C:\Users\USER\AppData\Local\Temp\ipykernel_1812\4107702595.py:3: DtypeWarning: Columns (0,13) have mixed types. Specify dtype option on import or set low_memory=False.
  panel = pd.read_csv("D:\\ENSAE Paris\\3A\\S1\\ML_Climate_risk\\PROJET\\29f9885375e0bb8047745286a114a491\\catnat_drought\\catnat_drought\\panel_final.csv", sep=";")


,N° Insee,Nom de la commune,event_period,Decision_text,recognized,Dernier_lag_decision,n_events,nbr_demandes_passees,nbr_demandes_passees_acceptées,avg_duration_days,...,NB_DEP,dep_recog_last5,Catnat_intensity_dep,accept_rate_dep,dep_ever_recog,dep_delay_median,total_dep_requests,years_all_communes_recognized_before,argile_niveau,swi_unif_min
0,1004,AMBERIEU EN BUGEY,2018-07,Reconnue,1,Non reconnue,1,0,0,183.0,...,391,3.0,3.0,0.048387,0.434783,534.0,70.0,0.0,1.0,0.221
1,1004,AMBERIEU EN BUGEY,2020-04,Reconnue,1,Reconnue,2,1,1,182.0,...,391,123.0,25.0,0.820000,0.434783,443.0,237.0,0.0,1.0,0.563
2,1004,AMBERIEU EN BUGEY,2022-01,Reconnue,1,Reconnue,3,2,2,272.0,...,391,151.0,0.0,0.904192,0.434783,412.0,331.0,0.0,1.0,0.972
3,1004,AMBERIEU EN BUGEY,2023-01,Reconnue,1,Reconnue,4,3,3,89.0,...,391,168.0,85.0,0.918033,0.434783,412.0,416.0,0.0,1.0,0.910
4,1007,AMBRONAY,2015-07,Non reconnue,0,Non reconnue,1,0,0,91.0,...,391,0.0,0.0,0.000000,0.434783,568.0,2.0,0.0,1.0,0.092


In [19]:
import geopandas as gpd

LimitAdmin = r"D:\ENSAE Paris\3A\S1\ML_Climate_risk\Nouveau dossier\Bases\ADE_4-0_GPKG_LAMB93_FXX-ED2025-10-15.gpkg"
gdf_commune = gpd.read_file(LimitAdmin, layer="commune").to_crs(2154)



In [20]:
# Convertir en string
panel["N° Insee"] = panel["N° Insee"].astype(str)

# Normaliser en 5 chiffres (ex: 1004 -> 01004)
panel["code_insee_clean"] = panel["N° Insee"].str.zfill(5)


In [23]:
gdf_commune = gpd.read_file(LimitAdmin, layer="commune").to_crs(2154)

# On garde uniquement ce qui est utile
gdf_commune = gdf_commune[["code_insee", "geometry"]].copy()

# ------------------------------------------------------------
# 2. Calcul des centroïdes (à l'intérieur du polygone)
# ------------------------------------------------------------
# centroid classique (peut être en dehors)
#gdf_commune["centroid"] = gdf_commune.geometry.centroid

# centroid garanti à l'intérieur du polygone
gdf_commune["centroid"] = gdf_commune.geometry.representative_point()

# Extraire coordonnées
gdf_commune["centroid_x"] = gdf_commune["centroid"].x
gdf_commune["centroid_y"] = gdf_commune["centroid"].y

# Convertir en strings pour merge propre
gdf_commune["code_insee"] = gdf_commune["code_insee"].astype(str)

# ------------------------------------------------------------
# 3. Merge avec ton panel
# ------------------------------------------------------------
panel1 = panel.copy()  # ton panel actuel
panel1["code_insee_clean"] = panel1["code_insee_clean"].astype(str)

# jointure
panel_with_centroids = panel1.merge(
    gdf_commune[["code_insee", "centroid_x", "centroid_y"]],
    left_on="code_insee_clean",
    right_on="code_insee",
    how="left"
)

# ------------------------------------------------------------
# 4. Résultat final
# ------------------------------------------------------------
panel_with_centroids.head()


,N° Insee,Nom de la commune,event_period,Decision_text,recognized,Dernier_lag_decision,n_events,nbr_demandes_passees,nbr_demandes_passees_acceptées,avg_duration_days,...,dep_ever_recog,dep_delay_median,total_dep_requests,years_all_communes_recognized_before,argile_niveau,swi_unif_min,code_insee_clean,code_insee,centroid_x,centroid_y
0,1004,AMBERIEU EN BUGEY,2018-07,Reconnue,1,Non reconnue,1,0,0,183.0,...,0.434783,534.0,70.0,0.0,1.0,0.221,01004,01004,883369.647078,6542480.6
1,1004,AMBERIEU EN BUGEY,2020-04,Reconnue,1,Reconnue,2,1,1,182.0,...,0.434783,443.0,237.0,0.0,1.0,0.563,01004,01004,883369.647078,6542480.6
2,1004,AMBERIEU EN BUGEY,2022-01,Reconnue,1,Reconnue,3,2,2,272.0,...,0.434783,412.0,331.0,0.0,1.0,0.972,01004,01004,883369.647078,6542480.6
3,1004,AMBERIEU EN BUGEY,2023-01,Reconnue,1,Reconnue,4,3,3,89.0,...,0.434783,412.0,416.0,0.0,1.0,0.910,01004,01004,883369.647078,6542480.6
4,1007,AMBRONAY,2015-07,Non reconnue,0,Non reconnue,1,0,0,91.0,...,0.434783,568.0,2.0,0.0,1.0,0.092,01007,01007,882262.861838,6548290.2


In [25]:
# Supprimer les colonnes inutiles
panel_with_centroids = panel_with_centroids.drop(columns=["code_insee_clean", "code_insee"], errors="ignore")

# Afficher pour vérifier
panel_with_centroids.head()


,N° Insee,Nom de la commune,event_period,Decision_text,recognized,Dernier_lag_decision,n_events,nbr_demandes_passees,nbr_demandes_passees_acceptées,avg_duration_days,...,Catnat_intensity_dep,accept_rate_dep,dep_ever_recog,dep_delay_median,total_dep_requests,years_all_communes_recognized_before,argile_niveau,swi_unif_min,centroid_x,centroid_y
0,1004,AMBERIEU EN BUGEY,2018-07,Reconnue,1,Non reconnue,1,0,0,183.0,...,3.0,0.048387,0.434783,534.0,70.0,0.0,1.0,0.221,883369.647078,6542480.6
1,1004,AMBERIEU EN BUGEY,2020-04,Reconnue,1,Reconnue,2,1,1,182.0,...,25.0,0.820000,0.434783,443.0,237.0,0.0,1.0,0.563,883369.647078,6542480.6
2,1004,AMBERIEU EN BUGEY,2022-01,Reconnue,1,Reconnue,3,2,2,272.0,...,0.0,0.904192,0.434783,412.0,331.0,0.0,1.0,0.972,883369.647078,6542480.6
3,1004,AMBERIEU EN BUGEY,2023-01,Reconnue,1,Reconnue,4,3,3,89.0,...,85.0,0.918033,0.434783,412.0,416.0,0.0,1.0,0.910,883369.647078,6542480.6
4,1007,AMBRONAY,2015-07,Non reconnue,0,Non reconnue,1,0,0,91.0,...,0.0,0.000000,0.434783,568.0,2.0,0.0,1.0,0.092,882262.861838,6548290.2


In [29]:
panel_with_centroids.to_csv("D:\\ENSAE Paris\\3A\\S1\\ML_Climate_risk\\PROJET\\29f9885375e0bb8047745286a114a491\\catnat_drought\\catnat_drought\\panel_final_centroides.csv", sep =";", index=False, encoding="utf-8")


In [24]:
#gdf_commune = gpd.read_file(LimitAdmin, layer="commune")
print(gdf_commune.crs)
#gdf_commune.sample(5)


EPSG:2154
